# PneumoScan -- Explainability: Grad-CAM & LIME

This notebook generates **visual explanations** for model predictions using two complementary techniques:

| Technique | Type | What it shows |
|---|---|---|
| **Grad-CAM** | Gradient-based, model-specific | Highlights the spatial regions the last convolutional layer attends to for a given class |
| **LIME** | Perturbation-based, model-agnostic | Identifies superpixel regions that most influence the predicted class |

We also include a **TFLite export** section at the end to prepare the best model for mobile / edge deployment.

---

## 1 -- Setup

In [ ]:
# === Colab Setup ===
import os, sys

# Mount Google Drive & clone repo (only runs in Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    REPO_DIR = '/content/PneumoScan'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/muhammadrakib2299/PneumoScan.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull
    
    sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
    print("Running on Google Colab")
except ImportError:
    sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))
    print("Running locally")

# === Imports ===
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

import config
from data_loader import load_test_dataset
from predict import get_best_model_path
from gradcam import (
    generate_gradcam,
    overlay_heatmap,
    generate_multiclass_gradcam,
    generate_gradcam_for_samples,
    plot_gradcam_grid,
)
from lime_explain import (
    create_lime_explainer,
    explain_image,
    generate_lime_for_samples,
    plot_lime_explanation,
    plot_gradcam_vs_lime,
)
from utils import save_figure

print(f"TensorFlow : {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
# Create all required output directories
config.ensure_dirs()

## 2 -- Load Best Model & Test Data

In [ ]:
best_path, best_name = get_best_model_path()
print(f"Best model : {best_name}")
print(f"Path       : {best_path}")

model = keras.models.load_model(best_path)
model.summary()

In [ ]:
test_ds = load_test_dataset()
print("Test dataset loaded.")

### Helper -- Collect Sample Images

We extract a few images per class from the test set for use in all visualisation sections below.

In [ ]:
N_PER_CLASS = 5

class_images = {i: [] for i in range(config.NUM_CLASSES)}
class_labels = {i: [] for i in range(config.NUM_CLASSES)}

for images, labels in test_ds:
    for img, lbl in zip(images.numpy(), labels.numpy()):
        cls = np.argmax(lbl)
        if len(class_images[cls]) < N_PER_CLASS:
            class_images[cls].append(img)
            class_labels[cls].append(cls)
    if all(len(v) >= N_PER_CLASS for v in class_images.values()):
        break

print("Collected sample images:")
for i, name in enumerate(config.CLASS_NAMES):
    print(f"  {name}: {len(class_images[i])} images")

---

## 3 -- Grad-CAM

### Theory

**Gradient-weighted Class Activation Mapping (Grad-CAM)** produces a coarse localisation heatmap by:

1. Computing the gradient of the target class score with respect to the feature maps of the **last convolutional layer**.  
2. Global-average-pooling those gradients to obtain per-channel importance weights.  
3. Taking a weighted combination of the feature maps, followed by ReLU, to produce a heatmap that highlights regions the network considers important for the target class.

$$\alpha_k^c = \frac{1}{Z} \sum_i \sum_j \frac{\partial y^c}{\partial A_{ij}^k}, \qquad L_{\text{Grad-CAM}}^c = \text{ReLU}\!\left(\sum_k \alpha_k^c A^k\right)$$

**Advantages**: fast to compute, architecture-aware, highlights class-discriminative regions.  
**Limitations**: resolution is limited by the spatial size of the last conv layer; cannot explain non-convolutional architectures.

### 3.1 -- Batch Grad-CAM Generation (5 per class)

In [ ]:
generate_gradcam_for_samples(model, test_ds, best_name, n_per_class=5)

### 3.2 -- Individual Grad-CAM Grids

For each class, we show the first sample image alongside its per-class Grad-CAM heatmaps.

In [ ]:
for cls_idx, cls_name in enumerate(config.CLASS_NAMES):
    img = class_images[cls_idx][0]
    gradcam_results = generate_multiclass_gradcam(model, img)
    plot_gradcam_grid(img, gradcam_results, f"{best_name} | True: {cls_name}")

---

## 4 -- LIME

### Theory

**LIME (Local Interpretable Model-agnostic Explanations)** explains *any* classifier by:

1. Segmenting the image into **superpixels** (contiguous regions of similar pixels).  
2. Randomly masking subsets of superpixels to create **perturbed** versions of the image.  
3. Querying the model on all perturbed images.  
4. Fitting a **sparse linear model** (LASSO) to the (perturbation, prediction) pairs, weighted by proximity to the original image.  
5. The learned linear weights indicate which superpixels contribute **positively** or **negatively** to each class prediction.

**Advantages**: model-agnostic, provides both positive and negative evidence, interpretable linear proxy.  
**Limitations**: slower (requires many forward passes per image), explanations can vary between runs, sensitive to superpixel segmentation.

### 4.1 -- Batch LIME Generation (3 per class)

In [ ]:
generate_lime_for_samples(model, test_ds, best_name, n_per_class=3)

### 4.2 -- Individual LIME Explanations

In [ ]:
explainer = create_lime_explainer()

for cls_idx, cls_name in enumerate(config.CLASS_NAMES):
    img = class_images[cls_idx][0]
    explanation = explain_image(model, img, explainer=explainer, num_samples=500)
    plot_lime_explanation(img, explanation, f"{best_name} | True: {cls_name}",
                         predicted_class=cls_idx)

---

## 5 -- Grad-CAM vs LIME: Side-by-Side Comparison

For 3 sample images (one per class), we display the original image, the Grad-CAM overlay, and the LIME explanation together so we can compare what each technique highlights.

In [ ]:
for cls_idx, cls_name in enumerate(config.CLASS_NAMES):
    img = class_images[cls_idx][0]

    # Grad-CAM
    heatmap = generate_gradcam(model, img, class_index=cls_idx)
    gradcam_overlay = overlay_heatmap(img, heatmap)

    # LIME
    lime_explanation = explain_image(model, img, explainer=explainer, num_samples=500)

    # Side-by-side
    save_path = os.path.join(
        config.COMPARISON_DIR,
        f"gradcam_vs_lime_{cls_name}_{best_name}.png",
    )
    plot_gradcam_vs_lime(
        img, gradcam_overlay, lime_explanation, cls_idx,
        model_name=f"{best_name} | {cls_name}",
        save_path=save_path,
    )

print("Side-by-side comparisons saved.")

---

## 6 -- Cross-Model Grad-CAM Comparison

We take a single test image and generate the Grad-CAM heatmap from each of the 5 trained models.  
This reveals how different architectures attend to different spatial regions.

In [ ]:
# Pick a sample image (first BACTERIA image)
sample_img = class_images[1][0]  # BACTERIA class
sample_class_idx = 1

fig, axes = plt.subplots(1, 1 + len(config.MODEL_NAMES), figsize=(5 * (1 + len(config.MODEL_NAMES)), 5))

# Original
img_display = sample_img if sample_img.max() <= 1.0 else sample_img / 255.0
axes[0].imshow(img_display)
axes[0].set_title("Original\n(BACTERIA)", fontsize=11, fontweight="bold")
axes[0].axis("off")

for i, model_name in enumerate(config.MODEL_NAMES):
    model_path = config.MODEL_SAVE_PATHS[model_name]
    if not os.path.exists(model_path):
        axes[i + 1].set_title(f"{model_name}\n(not found)", fontsize=11)
        axes[i + 1].axis("off")
        continue

    m = keras.models.load_model(model_path)
    heatmap = generate_gradcam(m, sample_img, class_index=sample_class_idx)
    overlay = overlay_heatmap(sample_img, heatmap)
    axes[i + 1].imshow(overlay)
    axes[i + 1].set_title(model_name, fontsize=11, fontweight="bold")
    axes[i + 1].axis("off")

fig.suptitle("Cross-Model Grad-CAM Comparison (BACTERIA sample)", fontsize=14, fontweight="bold")
plt.tight_layout()
save_figure(fig, os.path.join(config.COMPARISON_DIR, "cross_model_gradcam.png"))

---

## 7 -- Clinical Relevance of Explainable AI (XAI)

### Why Explainability Matters in Medical Imaging

Deep learning models for chest X-ray classification are often treated as **black boxes**.  
In clinical settings, this is unacceptable for several reasons:

1. **Trust**: Radiologists will not adopt a tool they cannot understand. Showing *where* the model looks builds trust.  
2. **Error detection**: If Grad-CAM highlights the image border or a text annotation instead of the lung fields, the clinician immediately knows the prediction is unreliable.  
3. **Regulatory compliance**: Frameworks such as the EU AI Act and FDA guidance for Software as a Medical Device (SaMD) increasingly require algorithmic transparency.  
4. **Educational value**: Explanations can help junior radiologists learn which visual patterns differentiate bacterial from viral pneumonia.

### Grad-CAM vs LIME in Practice

| Aspect | Grad-CAM | LIME |
|---|---|---|
| Speed | Fast (single backward pass) | Slow (hundreds of forward passes) |
| Resolution | Coarse (conv layer size) | Superpixel-level |
| Model dependency | Needs CNN gradients | Model-agnostic |
| Deterministic | Yes | Stochastic (varies per run) |
| Best for | Quick spatial localisation | Detailed feature importance |

**Recommendation**: use Grad-CAM for real-time visual feedback in production, and LIME for offline auditing and detailed case review.

---

## 8 -- TFLite Export

We convert the best model to **TensorFlow Lite** format for deployment on mobile devices or edge hardware (e.g., Raspberry Pi, Android).  
Two variants are exported:

1. **Float16** -- standard TFLite conversion (smaller than .keras, same accuracy)  
2. **Int8 quantised** -- further size reduction with minimal accuracy loss

In [ ]:
# Reload best model cleanly
best_path, best_name = get_best_model_path()
best_model = keras.models.load_model(best_path)
print(f"Loaded: {best_name} from {best_path}")

In [ ]:
os.makedirs(config.TFLITE_DIR, exist_ok=True)

# --- Standard (float16) conversion ---
converter_f16 = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter_f16.optimizations = [tf.lite.Optimize.DEFAULT]
converter_f16.target_spec.supported_types = [tf.float16]

tflite_f16 = converter_f16.convert()

f16_path = os.path.join(config.TFLITE_DIR, f"{best_name}_float16.tflite")
with open(f16_path, "wb") as f:
    f.write(tflite_f16)

print(f"Float16 TFLite saved: {f16_path}")
print(f"  Size: {os.path.getsize(f16_path) / (1024 * 1024):.2f} MB")

In [ ]:
# --- Int8 quantised conversion ---
# Representative dataset generator for calibration
def representative_dataset_gen():
    """Yield batches of test images for int8 calibration."""
    for images, _ in test_ds.take(10):
        for img in images:
            yield [tf.expand_dims(img, axis=0)]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset_gen
converter_int8.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
]
converter_int8.inference_input_type = tf.uint8
converter_int8.inference_output_type = tf.uint8

tflite_int8 = converter_int8.convert()

int8_path = os.path.join(config.TFLITE_DIR, f"{best_name}_int8.tflite")
with open(int8_path, "wb") as f:
    f.write(tflite_int8)

print(f"Int8 TFLite saved: {int8_path}")
print(f"  Size: {os.path.getsize(int8_path) / (1024 * 1024):.2f} MB")

In [ ]:
# --- File size comparison ---
keras_size = os.path.getsize(best_path) / (1024 * 1024)
f16_size = os.path.getsize(f16_path) / (1024 * 1024)
int8_size = os.path.getsize(int8_path) / (1024 * 1024)

print(f"\n{'=' * 50}")
print(f"Model Size Comparison ({best_name})")
print(f"{'=' * 50}")
print(f"  Keras (.keras)   : {keras_size:7.2f} MB")
print(f"  TFLite (float16) : {f16_size:7.2f} MB  ({f16_size / keras_size:.1%} of original)")
print(f"  TFLite (int8)    : {int8_size:7.2f} MB  ({int8_size / keras_size:.1%} of original)")

### 8.1 -- Test TFLite Inference

In [ ]:
import time

# Load float16 TFLite model
interpreter = tf.lite.Interpreter(model_path=f16_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("TFLite Input details:")
print(f"  Shape: {input_details[0]['shape']}")
print(f"  Dtype: {input_details[0]['dtype']}")
print(f"TFLite Output details:")
print(f"  Shape: {output_details[0]['shape']}")
print(f"  Dtype: {output_details[0]['dtype']}")

In [ ]:
# Run inference on a sample image
sample_img = class_images[0][0]  # First NORMAL image
input_data = np.expand_dims(sample_img, axis=0).astype(np.float32)

# Warm-up
interpreter.set_tensor(input_details[0]['index'], input_data)
interpreter.invoke()

# Timed inference
n_runs = 20
start = time.time()
for _ in range(n_runs):
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
elapsed = (time.time() - start) / n_runs * 1000  # ms

output_data = interpreter.get_tensor(output_details[0]['index'])
predicted_idx = np.argmax(output_data[0])
confidence = output_data[0][predicted_idx]

print(f"\nTFLite Inference Result:")
print(f"  Predicted class : {config.CLASS_NAMES[predicted_idx]}")
print(f"  Confidence      : {confidence:.4f}")
print(f"  Probabilities   : {dict(zip(config.CLASS_NAMES, output_data[0].tolist()))}")
print(f"  Avg latency     : {elapsed:.2f} ms/image ({n_runs} runs)")

In [ ]:
# Compare Keras vs TFLite prediction on the same image
keras_preds = best_model.predict(input_data, verbose=0)[0]
tflite_preds = output_data[0]

print("Prediction Comparison (same sample image):")
print(f"  {'Class':12s}  {'Keras':>10s}  {'TFLite':>10s}  {'Diff':>10s}")
print(f"  {'-' * 46}")
for i, name in enumerate(config.CLASS_NAMES):
    diff = abs(float(keras_preds[i]) - float(tflite_preds[i]))
    print(f"  {name:12s}  {keras_preds[i]:10.6f}  {tflite_preds[i]:10.6f}  {diff:10.6f}")

print(f"\nKeras predicted    : {config.CLASS_NAMES[np.argmax(keras_preds)]}")
print(f"TFLite predicted   : {config.CLASS_NAMES[np.argmax(tflite_preds)]}")
print(f"Predictions match  : {np.argmax(keras_preds) == np.argmax(tflite_preds)}")

---

## Summary

In this notebook we:

1. Generated **Grad-CAM** heatmaps for 5 images per class, showing which lung regions drive predictions.  
2. Generated **LIME** explanations for 3 images per class, identifying superpixel-level positive and negative evidence.  
3. Compared Grad-CAM and LIME **side-by-side** for one image per class -- both methods typically agree on the lung parenchyma as the primary region of interest.  
4. Performed a **cross-model Grad-CAM comparison**, revealing that different architectures attend to overlapping but not identical regions.  
5. Exported the best model to **TFLite** (float16 and int8), verified that predictions match, and reported file sizes and latency.

These explainability artifacts and the TFLite models are saved under `outputs/` for use in the deployment pipeline.